**채용 공고 1건을 여러 관점에서 동시에 분석하고, 지원자 정보와 비교해 매칭 결과를 반환하는 LCEL 파이프라인을 만들어라.**
```
posting = """[백엔드 개발자] 결제 서비스 API 설계 및 운영. 경력 3년 이상.
필수: Python, FastAPI, PostgreSQL, Docker / 우대: Kubernetes, AWS"""
candidate = {"name": "지원자A", "years": 4, "skills": ["Python", "Django", "PostgreSQL", "Docker", "AWS"]}
RunnableParallel로 세 체인을 동시에 실행한다.
```
* skills: 필수 기술 스택 추출. JSON({"skills": [...]})으로 받아 itemgetter("skills")로 리스트만 꺼낸다.

* career: 요구 경력(년)을 숫자로 추출

* summary: 담당 업무 한 문장 요약

* RunnablePassthrough.assign으로 입력(posting, candidate)을 보존한 채 분석 결과를 analysis 키로 추가한다.

* RunnableLambda로 지원자 기술과 필수 기술을 코드로 비교해 다음을 계산한다.

    * 매칭 점수: 일치 기술 수 / 필수 기술 수 × 100

    * 일치 · 부족 기술 목록

    * 경력 충족 여부

* 같은 세 체인을 순차 실행했을 때와 병렬 실행 시간을 비교해 출력한다.

In [29]:
import re
from operator import itemgetter

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

load_dotenv()
llm = ChatOpenAI(model="gpt-6-luna")

In [30]:
# 테스트 데이터
posting = """[백엔드 개발자] 결제 서비스 API 설계 및 운영. 경력 3년 이상.
필수: Python, FastAPI, PostgreSQL, Docker / 우대: Kubernetes, AWS"""
candidate = {"name": "지원자A", "years": 4, "skills": ["Python", "Django", "PostgreSQL", "Docker", "AWS"]}

In [31]:
# skills: 필수 기술 스택 추출 (JSON -> itemgetter로 리스트만 추출)
skills_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "채용 공고에서 필수 기술 스택만 추출함. 우대 기술은 제외함. "
         '반드시 JSON으로만 답함: {{"skills": ["기술1", "기술2"]}}'),
        ("human", "{posting}"),
    ])
    | llm
    | JsonOutputParser()
    | itemgetter("skills")
)

# career: 요구 경력(년) 숫자 추출
career_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "채용 공고의 요구 경력(년)을 숫자만 답함. 예: 3"),
        ("human", "{posting}"),
    ])
    | llm
    | StrOutputParser()
    | RunnableLambda(lambda s: int(re.search(r"\d+", s).group()))
)

# summary: 담당 업무 한 문장 요약
summary_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "채용 공고의 담당 업무를 한 문장으로 요약함."),
        ("human", "{posting}"),
    ])
    | llm
    | StrOutputParser()
)

chains = {"skills": skills_chain, "career": career_chain, "summary": summary_chain}

In [32]:
# RunnableParallel로 세 체인 동시 실행
analysis_parallel = RunnableParallel(chains)

In [33]:
# 입력 보존 + analysis 키 추가 후 코드로 매칭 계산
def match(inp):
    cand = inp["candidate"]
    analysis = inp["analysis"]
    required = analysis["skills"]
    have = {s.lower() for s in cand["skills"]}

    matched = [s for s in required if s.lower() in have]
    missing = [s for s in required if s.lower() not in have]
    score = len(matched) / len(required) * 100 if required else 0.0

    return {
        "name": cand["name"],
        "summary": analysis["summary"],
        "score": round(score, 1),
        "matched": matched,
        "missing": missing,
        "career_ok": cand["years"] >= analysis["career"],
    }


pipeline = RunnablePassthrough.assign(analysis=analysis_parallel) | RunnableLambda(match)

In [34]:
# 파이프라인 실행 및 매칭 결과 출력
result = pipeline.invoke({"posting": posting, "candidate": candidate})

print(
    f"지원자: {result['name']}\n"
    f"업무 요약: {result['summary']}\n"
    f"매칭 점수: {result['score']}점\n"
    f"일치 기술: {result['matched']}\n"
    f"부족 기술: {result['missing']}\n"
    f"경력 충족: {'O' if result['career_ok'] else 'X'}"
)

지원자: 지원자A
업무 요약: 결제 서비스 API를 설계하고 안정적으로 운영합니다.
매칭 점수: 75.0점
일치 기술: ['Python', 'PostgreSQL', 'Docker']
부족 기술: ['FastAPI']
경력 충족: O


In [35]:
# 순차 실행과 병렬 실행 시간 비교
import time

inp = {"posting": posting, "candidate": candidate}

start = time.perf_counter()
seq_result = {name: chain.invoke(inp) for name, chain in chains.items()}
seq_time = time.perf_counter() - start

start = time.perf_counter()
par_result = analysis_parallel.invoke(inp)
par_time = time.perf_counter() - start

print(f"순차 실행: {seq_time:.2f}초")
print(f"병렬 실행: {par_time:.2f}초")
print(f"속도 비율: {seq_time / par_time:.2f}배")

순차 실행: 4.24초
병렬 실행: 1.87초
속도 비율: 2.27배


---

**입력 문장의 언어를 감지해 알맞은 번역 체인으로 보내는 번역봇을 만들어라.**

* RunnableLambda로 언어를 감지한다. 한글 · 영문 글자 수를 정규식으로 세어 {"text", "lang"}을 반환하며, LLM을 쓰지 않는다.

* RunnableBranch로 라우팅한다.

* lang == "ko" → 한→영 번역 체인

* lang == "en" → 영→한 번역 체인

* 그 외(숫자 · 기호만 있는 입력) → 번역 불가 안내 메시지 (기본 분기)

* 번역 모델은 gpt-6.1-sol을 주 모델로, gpt-6-luna를 with_fallbacks의 대체 모델로 설정한다.

* 결과를 [언어] 원문 → 번역문 형식으로 만드는 후처리 함수를 붙이고, 전체 체인에 with_retry를 적용한다.

* 한국어 · 영어 · 숫자만 있는 입력 · 앞뒤 공백이 있는 입력 4개를 batch로 처리한다.

In [36]:
import re

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnablePassthrough

load_dotenv()
# 주 모델 gpt-6.1-sol, 대체 모델 gpt-6-luna
llm = ChatOpenAI(model="gpt-6.1-sol").with_fallbacks([ChatOpenAI(model="gpt-6-luna")])

In [37]:
# 언어 감지 (정규식, LLM 미사용)
def detect_lang(text):
    text = text.strip()
    ko = len(re.findall(r"[가-힣]", text))
    en = len(re.findall(r"[A-Za-z]", text))
    if ko == 0 and en == 0:
        lang = "other"
    elif ko >= en:
        lang = "ko"
    else:
        lang = "en"
    return {"text": text, "lang": lang}


detector = RunnableLambda(detect_lang)

In [38]:
# 번역 체인 (한->영, 영->한)
ko2en = (
    ChatPromptTemplate.from_messages([
        ("system", "한국어 문장을 영어로 번역함. 번역문만 출력함."),
        ("human", "{text}"),
    ])
    | llm
    | StrOutputParser()
)

en2ko = (
    ChatPromptTemplate.from_messages([
        ("system", "영어 문장을 한국어로 번역함. 번역문만 출력함."),
        ("human", "{text}"),
    ])
    | llm
    | StrOutputParser()
)

In [39]:
# RunnableBranch 라우팅 (마지막 인자가 기본 분기)
fallback_message = RunnableLambda(lambda x: "번역 불가: 한글 또는 영문이 포함된 문장만 번역할 수 있음")

branch = RunnableBranch(
    (lambda x: x["lang"] == "ko", ko2en),
    (lambda x: x["lang"] == "en", en2ko),
    fallback_message,
)

In [40]:
# 후처리([언어] 원문 -> 번역문) 및 전체 체인, with_retry 적용
LANG_LABEL = {"ko": "한국어", "en": "영어"}


def format_result(x):
    label = LANG_LABEL.get(x["lang"], "알 수 없음")
    return f"[{label}] {x['text']} → {x['translated']}"


chain = (
    detector
    | RunnablePassthrough.assign(translated=branch)
    | RunnableLambda(format_result)
).with_retry(stop_after_attempt=3)

In [41]:
# 4개 입력 batch 처리 (한국어, 영어, 숫자만, 앞뒤 공백)
inputs = [
    "안녕하세요, 오늘 날씨가 정말 좋네요.",
    "Nice to meet you. I am learning LangChain.",
    "12345 !@#$%",
    "   데이터 분석은 정말 재미있어요   ",
]

for output in chain.batch(inputs):
    print(output)

[한국어] 안녕하세요, 오늘 날씨가 정말 좋네요. → Hello, the weather is really nice today.
[영어] Nice to meet you. I am learning LangChain. → 만나서 반갑습니다. 저는 LangChain을 배우고 있습니다.
[알 수 없음] 12345 !@#$% → 번역 불가: 한글 또는 영문이 포함된 문장만 번역할 수 있음
[한국어] 데이터 분석은 정말 재미있어요 → Data analysis is really fun.


---

**사용자(세션)별로 대화를 기억하는 학습 도우미 챗봇을 만들고, 대화가 길어져도 모델에는 최근 대화만 전달되도록 하라.**

* 프롬프트: system({subject} 학습 도우미, 2문장 설명 + 확인 질문 1개) + MessagesPlaceholder("history") + human({question})

* trim_messages로 최근 메시지 6개만 남기는 trimmer를 만든다(token_counter=len, strategy="last", start_on="human").

* RunnablePassthrough.assign(history=itemgetter("history") | trimmer)로 프롬프트 앞단에서 기록을 자른다.

* RunnableWithMessageHistory로 세션별 기록을 저장하고, with_config로 run_name과 tags를 지정한다.

* 답변은 stream으로 출력하고, 호출마다 저장소에 쌓인 메시지 수를 출력한다.

* 테스트: 세션 2개를 섞어 대화하고, 한 세션에서 5턴 이상 대화한 뒤 첫 턴에서 말한 이름을 물어본다.

In [42]:
from operator import itemgetter

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import trim_messages
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory

load_dotenv()
llm = ChatOpenAI(model="gpt-6-luna")

In [43]:
# 프롬프트: system + history + human
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {subject} 학습 도우미임. 개념을 2문장으로 설명한 뒤 이해를 확인하는 질문을 1개 함."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

In [44]:
# 최근 메시지 6개만 남기는 trimmer
trimmer = trim_messages(
    max_tokens=6,
    token_counter=len,
    strategy="last",
    start_on="human",
)

In [45]:
# 프롬프트 앞단에서 기록을 자르는 체인
chain = (
    RunnablePassthrough.assign(history=itemgetter("history") | trimmer)
    | prompt
    | llm
    | StrOutputParser()
)

In [46]:
# 세션별 기록 저장소 + RunnableWithMessageHistory (저장은 전체, 모델 전달은 trimmer가 처리)
store = {}


def get_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


chat = RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key="question",
    history_messages_key="history",
).with_config(run_name="study_helper", tags=["study-bot"])

c:\Users\kevin\campus\TIL-kevin\workspace\12_GenAI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [47]:
# stream 출력 및 호출마다 저장된 메시지 수 출력
def ask(session_id, question, subject):
    print(f"[{session_id}] Q: {question}")
    print("A: ", end="")
    for chunk in chat.stream(
        {"subject": subject, "question": question},
        config={"configurable": {"session_id": session_id}},
    ):
        print(chunk, end="", flush=True)
    print(f"\n  -> 저장된 메시지 수: {len(get_history(session_id).messages)}\n")

In [48]:
# 세션 2개를 섞어 대화 (A: 6턴, B: 3턴)
ask("A", "안녕, 내 이름은 민수야. 파이썬 리스트가 뭐야?", "파이썬")
ask("B", "나는 지은이라고 해. 데이터프레임이 뭐야?", "판다스")
ask("A", "튜플은 리스트와 뭐가 달라?", "파이썬")
ask("B", "결측치는 어떻게 처리해?", "판다스")
ask("A", "딕셔너리는 언제 써?", "파이썬")
ask("A", "집합은 어디에 유용해?", "파이썬")
ask("B", "groupby는 뭐하는 거야?", "판다스")
ask("A", "for문과 while문의 차이는?", "파이썬")
ask("A", "함수는 왜 만들어서 써?", "파이썬")

[A] Q: 안녕, 내 이름은 민수야. 파이썬 리스트가 뭐야?
A: 안녕, 민수야! 파이썬 리스트는 여러 값을 순서대로 담아 두는 자료형이야. 리스트 안의 값은 인덱스(위치 번호)로 꺼내거나 바꿀 수 있어.  
예를 들어 `["사과", "바나나"]`에서 `"바나나"`를 꺼낼 때는 어떤 인덱스를 사용하면 될까?
  -> 저장된 메시지 수: 2

[B] Q: 나는 지은이라고 해. 데이터프레임이 뭐야?
A: 지은님, 데이터프레임은 행과 열로 구성된 표 형태의 데이터 구조예요. 각 열에는 서로 다른 종류의 데이터가 들어갈 수 있고, 판다스에서는 `DataFrame`으로 다뤄요.

데이터프레임은 어떤 모양의 데이터라고 생각하면 될까요?
  -> 저장된 메시지 수: 2

[A] Q: 튜플은 리스트와 뭐가 달라?
A: 리스트는 항목을 추가하거나 바꿀 수 있지만, 튜플은 만든 뒤 항목을 바꿀 수 없어. 리스트는 `[]`, 튜플은 보통 `()`로 만들고, 예를 들면 `[1, 2]`와 `(1, 2)`처럼 써.  
튜플 `(1, 2)`의 첫 번째 값을 `3`으로 바꿀 수 있을까?
  -> 저장된 메시지 수: 4

[B] Q: 결측치는 어떻게 처리해?
A: 결측치는 판다스에서 `NaN` 등으로 표시하며, `df.isna()`로 어디에 있는지 확인할 수 있어요. 결측치가 있는 행이나 열은 `df.dropna()`로 삭제하고, `df.fillna(값)`으로 특정 값이나 평균값 등을 채울 수 있어요.

결측치가 있는 행을 삭제하려면 어떤 메서드를 쓰면 될까요?
  -> 저장된 메시지 수: 4

[A] Q: 딕셔너리는 언제 써?
A: 딕셔너리는 이름표(키)와 값(데이터)을 짝지어 저장할 때 써. 예를 들어 사람의 이름으로 전화번호를 찾거나, 상품명으로 가격을 조회할 때 편리해.  
`{"민수": 1234}`에서 전화번호를 찾을 때 사용하는 키는 무엇일까?
  -> 저장된 메시지 수: 6

[A] Q: 집합은 어디에 유용해?
A: 집합(`set`)은 중복된 값을 자동으로 하나로 모으고, 어

In [49]:
# 첫 턴의 이름 회상 확인
# A: 6턴 뒤라 최근 6개 메시지에 첫 턴이 없음 -> 이름을 모를 것으로 예상
# B: 3턴 뒤라 첫 턴이 남아 있음 -> 이름을 기억할 것으로 예상
ask("A", "내 이름이 뭐였는지 기억해?", "파이썬")
ask("B", "내 이름이 뭐였는지 기억해?", "판다스")

[A] Q: 내 이름이 뭐였는지 기억해?
A: 아직 이름을 알려주지 않아서 기억하고 있는 이름은 없어. 이름을 알려주면 이 대화에서 그렇게 부를게.  
어떤 이름으로 불러드릴까?
  -> 저장된 메시지 수: 14

[B] Q: 내 이름이 뭐였는지 기억해?
A: 지은이라고 하셨어요!
  -> 저장된 메시지 수: 8

